# Sporting Fronts

**Analytical purpose:** In which sports did the USA or USSR hold the medal advantage?

This notebook is the official chart-specific preprocessing pipeline. Shared Olympic/geography logic lives in `common.py`.

In [1]:
from pathlib import Path
import sys

CHARTS_DIR = Path.cwd()
if CHARTS_DIR.name != 'charts':
    candidates = [p / 'preprocessing' / 'charts' for p in [Path.cwd(), *Path.cwd().parents]]
    CHARTS_DIR = next((p for p in candidates if (p / 'common.py').exists()), None)
    if CHARTS_DIR is None:
        raise RuntimeError('Run this notebook from the repository or preprocessing/charts directory.')
sys.path.insert(0, str(CHARTS_DIR))
from common import *
ensure_output_dirs()

In [2]:
import pandas as pd

pulse = load_rivalry_matches()
pulse = pulse[(pulse["counts_for_pulse"] == True) & pulse["winner"].isin(["USA", "URS", "DRAW"]) & pulse["year"].isin(RIVALRY_YEARS)].copy()
head_to_head_keys = set(zip(pulse["year"].astype(int), pulse["sport"].astype(str)))

sport = medal_summary_by_sport()
sport = sport[sport.Year.isin(JOINT_YEARS) & sport.NOC.isin(["USA", "URS"])].copy()

records = []
def add_scope(scope, frame):
    pivot = frame.pivot_table(
        index=["Year", "Sport"], columns="NOC",
        values=["TotalMedals", "GoldMedals"], aggfunc="sum", fill_value=0
    )
    for (year, sport_name), row in pivot.iterrows():
        usa_total = int(row.get(("TotalMedals", "USA"), 0))
        ussr_total = int(row.get(("TotalMedals", "URS"), 0))
        usa_gold = int(row.get(("GoldMedals", "USA"), 0))
        ussr_gold = int(row.get(("GoldMedals", "URS"), 0))
        records.append({
            "Year": str(year), "Sport": sport_name, "Scope": scope,
            "USATotal": usa_total, "USSRTotal": ussr_total,
            "USAGold": usa_gold, "USSRGold": ussr_gold,
            "TotalDifference": usa_total - ussr_total,
            "GoldDifference": usa_gold - ussr_gold,
        })

    aggregate = (
        frame.groupby(["Sport", "NOC"], as_index=False)[["TotalMedals", "GoldMedals"]].sum()
        .pivot_table(index="Sport", columns="NOC", values=["TotalMedals", "GoldMedals"], aggfunc="sum", fill_value=0)
    )
    for sport_name, row in aggregate.iterrows():
        usa_total = int(row.get(("TotalMedals", "USA"), 0))
        ussr_total = int(row.get(("TotalMedals", "URS"), 0))
        usa_gold = int(row.get(("GoldMedals", "USA"), 0))
        ussr_gold = int(row.get(("GoldMedals", "URS"), 0))
        records.append({
            "Year": "ALL", "Sport": sport_name, "Scope": scope,
            "USATotal": usa_total, "USSRTotal": ussr_total,
            "USAGold": usa_gold, "USSRGold": ussr_gold,
            "TotalDifference": usa_total - ussr_total,
            "GoldDifference": usa_gold - ussr_gold,
        })

add_scope("all_games", sport)
add_scope("head_to_head", sport[sport.apply(lambda r: (int(r.Year), str(r.Sport)) in head_to_head_keys, axis=1)])

out = pd.DataFrame(records).sort_values(["Scope", "Year", "Sport"]).reset_index(drop=True)

In [3]:
sports_to_exclude = ["Modern Pentathlon", "Synchronized Swimming", "Equestrianism", "Rhythmic Gymnastics", "Handball"]
out = out[~out["Sport"].isin(sports_to_exclude)]

assert not out.Year.isin(["1980", "1984"]).any()


In [4]:

assert not out.Year.isin(["1980", "1984"]).any()
path = FINAL_DIR / "sporting_fronts.csv"
out.to_csv(path, index=False)
print(f"Wrote {path.relative_to(REPO_ROOT)}: {len(out)} rows")
out.head()

Wrote data\final\cold_war\sporting_fronts.csv: 202 rows


,Year,Sport,Scope,USATotal,USSRTotal,USAGold,USSRGold,TotalDifference,GoldDifference
0,1952,Athletics,all_games,31,17,15,2,14,13
1,1952,Basketball,all_games,1,1,1,0,0,1
2,1952,Boxing,all_games,5,6,5,0,-1,5
3,1952,Canoeing,all_games,1,1,1,0,0,1
4,1952,Diving,all_games,9,0,4,0,9,4
